# Pneumonia detection v2 — Kaggle runner

Runs the full pipeline on a Kaggle GPU: patient-level split → training → calibration → internal test +
**external RSNA test** → Grad-CAM checks → results table.

**Notebook settings (right panel):**
- *Input*: add the dataset `paultimothymooney/chest-xray-pneumonia` and the competition `rsna-pneumonia-detection-challenge`
- *Accelerator*: **GPU T4 x2** (the code uses one GPU)
- *Internet*: **on** (to install the package from GitHub)

Use **Save Version → Save & Run All (Commit)** so it runs in the background (~2.5–3 h for all five models).
Results appear in the *Output* tab as `results_bundle.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/empresst/PneumoniaDetectionExplainableAI.git repo
%cd /kaggle/working/repo
!pip install -q -e .

## 2. Locate the attached datasets (mount paths differ between Kaggle versions)

In [ ]:
from pathlib import Path

PEDIATRIC = next(p for p in Path('/kaggle/input').rglob('chest_xray') if p.is_dir())
RSNA = next(Path('/kaggle/input').rglob('stage_2_train_labels.csv')).parent
print('pediatric:', PEDIATRIC)
print('rsna:     ', RSNA)

## 3. Manifest (patient-level split) + image cache

The cache lives in `/tmp` so it is not copied into the notebook output.
`--rsna-n 6000` = stratified sample of 6,000 RSNA studies; `0` uses all ~26.7k.

In [ ]:
!mkdir -p /tmp/data && ln -sfn /tmp/data data
!python -m pneumo.prepare --pediatric "{PEDIATRIC}" --rsna "{RSNA}" --rsna-n 6000 --out data --workers 4

## 4. Main model + baselines/ablations (train → evaluate → explain), then the results table

In [ ]:
!bash scripts/run_all.sh

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('results/RESULTS.md').read()))
for subset in ['test', 'rsna_opacity_vs_normal']:
    display(Image(f'runs/densenet121_224/eval/{subset}/curves.png'))
    display(Image(f'runs/densenet121_224/eval/{subset}/gradcam_gallery.png'))

In [ ]:
import pandas as pd
pd.read_csv('runs/densenet121_224/eval/rsna_opacity_vs_all/subgroups.csv')

## 5. Bundle results (download from the Output tab)

In [ ]:
!zip -q -r /kaggle/working/results_bundle.zip results runs data/manifest.csv -x '*.pt'
!mkdir -p /kaggle/working/checkpoints && cp runs/densenet121_224/best.pt /kaggle/working/checkpoints/densenet121_224.pt
!ls -lh /kaggle/working